In [1]:
import pandas as pd

clean = pd.read_csv("../data/processed/clean_transactions.csv",
                    parse_dates=["InvoiceDate"], dtype={"StockCode": str})
print(clean.shape)

(1036877, 7)


In [ ]:
# Start with week from Monday
clean["week"] = clean["InvoiceDate"].dt.to_period("W-SUN").dt.start_time
print(clean["week"].min(), "to", clean["week"].max())

2009-11-30 00:00:00 to 2011-12-05 00:00:00


In [3]:
# remove first and last week, as they are incomplete
first_week = clean["week"].min()
last_week = clean["week"].max()
clean = clean[(clean["week"] > first_week) & (clean["week"] < last_week)]
print(clean["week"].min(), "to", clean["week"].max())

2009-12-07 00:00:00 to 2011-11-28 00:00:00


In [4]:
#Product week Table 
weekly = (clean.groupby(["StockCode", "week"])
               .agg(units=("Quantity", "sum"),
                    orders=("Invoice", "nunique"),
                    avg_price=("Price", "mean"))
               .reset_index())
print(weekly.shape)
weekly.head()

(191530, 5)


,StockCode,week,units,orders,avg_price
0,10002,2009-12-07,24,4,1.062500
1,10002,2009-12-14,37,3,1.133333
2,10002,2009-12-21,14,3,1.133333
3,10002,2010-01-04,15,2,0.850000
4,10002,2010-01-11,191,4,0.850000


In [5]:
# Make a calendar of each product
end = weekly["week"].max()
first_sale = weekly.groupby("StockCode")["week"].min()

grid = pd.concat([
    pd.DataFrame({"StockCode": code,
                  "week": pd.date_range(start, end, freq="7D")})
    for code, start in first_sale.items()
], ignore_index=True)

print(grid.shape)

(407702, 2)


In [6]:
# Then marge and fill with 0 
panel = grid.merge(weekly, on=["StockCode", "week"], how="left")
panel[["units", "orders"]] = panel[["units", "orders"]].fillna(0)

# price: pichla known price aage le jao (sirf past information)
panel = panel.sort_values(["StockCode", "week"])
panel["avg_price"] = panel.groupby("StockCode")["avg_price"].ffill()

print(panel.shape)
print(panel.isna().sum())


(407702, 5)
StockCode    0
week         0
units        0
orders       0
avg_price    0
dtype: int64


In [7]:
# Save 
panel.to_csv("../data/processed/weekly_panel.csv", index=False)
print(panel["StockCode"].nunique(), "products,", panel["week"].nunique(), "weeks")

4675 products, 104 weeks
